# WACV 2027 Workshop on Real-World Surveillance (RWS) — Track 2: ABPR
## EXP-006: Stage 2 Cross-Modal Metric Learning & ArcFace Alignment Training

> **Objective:** Train VisualProjector (768 -> 512) and QueryEncoder (40 -> 512) with ArcFace Additive Angular Margin Loss ($s=30, m=0.20$) over 7,204 identities and Symmetric InfoNCE Contrastive Loss to break the 40-bit crowding bottleneck and elevate Rank-1 retrieval into the championship tier (target: $\ge 0.65 - 0.68\text{ mADM}$).
> **Target Hardware:** Kaggle Dual Tesla T4 (32GB VRAM Total, 30GB CPU RAM) or Single GPU P100/T4.
> **Execution Mode:** Fast Stage 2 Decoupled Training (~15-20 minutes total runtime).


In [ ]:
# ==============================================================================
# 1. ENVIRONMENT VERIFICATION & REPOSITORY CLONING
# ==============================================================================
import os
import sys
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Clone or update RWS2027 codebase
WORKSPACE = "/kaggle/working/RWS2027"
if not os.path.exists(WORKSPACE):
    !git clone https://github.com/samankh93/RWS2027.git {WORKSPACE}
else:
    !cd {WORKSPACE} && git pull origin main

sys.path.insert(0, WORKSPACE)
os.chdir(WORKSPACE)
print(f"Active Working Directory: {os.getcwd()}")

In [ ]:
# ==============================================================================
# 2. DATASET LOCATION DETECTION & OPTIMIZATION
# ==============================================================================
import glob

dataset_search = glob.glob("/kaggle/input/**/annotations/train.csv", recursive=True)
if dataset_search:
    DATA_DIR = os.path.dirname(os.path.dirname(dataset_search[0]))
    print(f"Found UPAR dataset root at: {DATA_DIR}")
else:
    DATA_DIR = "/kaggle/input/datasets/hinewk/upar-challenge-2027-data"
    print(f"Defaulting to standard dataset path: {DATA_DIR}")


In [ ]:
# ==============================================================================
# 3. EXECUTE STAGE 2 ARCFACE METRIC LEARNING TRAINING
# ==============================================================================
!python train_stage2_metric.py \
    --model convnext_tiny \
    --submission_zip submissions/EXP-004_submission.zip \
    --data_root {DATA_DIR} \
    --epochs 30 \
    --batch_size 512 \
    --lr 0.001 \
    --scale 30.0 \
    --margin 0.20 \
    --lambda_contrastive 0.5 \
    --save_path checkpoints/stage2_convnext_metric_heads.pth

In [ ]:
# ==============================================================================
# 4. EXECUTE STAGE 2 TRAINING FOR SWIN-T
# ==============================================================================
!python train_stage2_metric.py \
    --model swin_t \
    --submission_zip submissions/EXP-005A_submission.zip \
    --data_root {DATA_DIR} \
    --epochs 30 \
    --batch_size 512 \
    --lr 0.001 \
    --scale 30.0 \
    --margin 0.20 \
    --lambda_contrastive 0.5 \
    --save_path checkpoints/stage2_swin_metric_heads.pth